# Hello World - Coupled Magnetoelastic Theory
## Magnetoelastic Hysteresis

This notebook demonstrates the code used to reproduce the results in: Sablik, M.J.; Jiles, D.C., "Coupled magnetoelastic theory of magnetic and magnetostrictive hysteresis," in Magnetics, IEEE Transactions on , vol.29, no.4, pp.2113-2123, Jul 1993. 

Approximate reproduction of stress-dependent magnetization hysteresis curves
similar to the figure shown in the 1993 IEEE Transactions on Magnetics paper.

This is a practical, tunable implementation based on:
    1. Jiles–Atherton-like magnetization hysteresis
    2. Stress added through an effective field term
    3. Magnetostriction computed from magnetization


B. Howard

## Libraries

In [17]:
#import sys
#!{sys.executable} -m pip install subprocess

In [18]:
import sys
import os

# Add parent directory to path to import magnetoelasticsensor package
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

### Do the libraries we need exist?

In [19]:
import subprocess
import sys
import math

# Install required packages
required_packages = [
    'numpy',
    'pandas',
    'matplotlib',
]

print("Checking and installing required libraries...")
print("=" * 70)

for package in required_packages:
    try:
        __import__(package)
        print(f"✓ {package:12} is already installed")
    except ImportError:
        print(f"  Installing {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
        print(f"✓ {package:12} installed successfully")

print("=" * 70)
print("✓ All required libraries are available!\n")

Checking and installing required libraries...
✓ numpy        is already installed
✓ pandas       is already installed
✓ matplotlib   is already installed
✓ All required libraries are available!



### Bring in the libraries

In [20]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from magnetoelasticsensor.anhyst_iso import (
    anhysteretic_magnetization,
    anhysteretic_magnetization_differential,
)
from magnetoelasticsensor.ja_solver import ja_solver
from magnetoelasticsensor.ja_props import JAProps


### Plot Controls

In [21]:
# Check for cairo backend availability and configure
pdf_backend = 'pdf'  # Default fallback
try:
    import cairo
    pdf_backend = 'cairo'
    print("✓ Cairo backend available - using for vector PDF output")
except ImportError:
    try:
        __import__('cairocffi')
        pdf_backend = 'cairo'
        print("✓ CairoCFFI backend available - using for vector PDF output")
    except ImportError:
        print("⚠ Cairo/CairoCFFI not available - using native PDF backend (vector-based)")

✓ Cairo backend available - using for vector PDF output


In [22]:
plt.rc('font', family='arial')
plt.rcParams.update({'font.size': 12})

In [23]:
from pathlib import Path
pdf_output_dir = Path('plots_pdf')
pdf_output_dir.mkdir(exist_ok=True)

### Helper function to generate smart filenames for plots

In [24]:
def generate_pdf_filename(section_name, plot_name):
    """
    Generate a smart PDF filename based on section and plot names.
    Converts spaces and special characters to underscores, removes extra dots.
    
    Parameters
    ----------
    section_name : str
        Section identifier (e.g., '6.1', '7.0')
    plot_name : str
        Descriptive plot name (e.g., 'target_permeance_vs_frequency')
    
    Returns
    -------
    Path
        Full path to the output PDF file
    """
    # Sanitize names: replace spaces and special chars with underscores
    section_clean = section_name.replace('.', '_').strip()
    plot_clean = plot_name.lower().replace(' ', '_').replace('-', '_')
    plot_clean = ''.join(c if c.isalnum() or c == '_' else '' for c in plot_clean)
    
    filename = f"section_{section_clean}_{plot_clean}.pdf"
    return pdf_output_dir / filename

In [25]:
def save_plot_pdf(fig, section_name, plot_name, dpi=300):
    """
    Save a matplotlib figure as a vector-based PDF file.
    
    Parameters
    ----------
    fig : matplotlib.figure.Figure
        Figure object to save
    section_name : str
        Section identifier (e.g., '6.1')
    plot_name : str
        Descriptive plot name
    dpi : int
        DPI for rasterization (default 300 for high quality)
    """
    pdf_path = generate_pdf_filename(section_name, plot_name)
    # Save as PDF using the configured backend
    fig.savefig(pdf_path, format='pdf', dpi=dpi, bbox_inches='tight')
    print(f"✓ PDF saved: {pdf_path}")
    return pdf_path

print(f"✓ PDF output directory: {pdf_output_dir.absolute()}")
print(f"✓ Smart filename generator and PDF saver ready\n")

✓ PDF output directory: c:\Local Docs\Github\MagnetoelasticSensor\notebooks\plots_pdf
✓ Smart filename generator and PDF saver ready



In [26]:
def generate_pdf_filename(section_name, plot_name):
    """
    Generate a smart PDF filename based on section and plot names.
    Converts spaces and special characters to underscores, removes extra dots.
    
    Parameters
    ----------
    section_name : str
        Section identifier (e.g., '6.1', '7.0')
    plot_name : str
        Descriptive plot name (e.g., 'target_permeance_vs_frequency')
    
    Returns
    -------
    Path
        Full path to the output PDF file
    """
    # Sanitize names: replace spaces and special chars with underscores
    section_clean = section_name.replace('.', '_').strip()
    plot_clean = plot_name.lower().replace(' ', '_').replace('-', '_')
    plot_clean = ''.join(c if c.isalnum() or c == '_' else '' for c in plot_clean)
    
    filename = f"section_{section_clean}_{plot_clean}.pdf"
    return pdf_output_dir / filename

### Constants

In [27]:
MU0 = JAProps.MU0  # H/m

### Functions

## 1. Plot the results

In [28]:
# Jiles-Atherton model parameters — shared across all plots in this notebook
props = JAProps(
    ms=1.6e6,
    a=1100,
    alpha=0.0016,
    k=400.0,
    c=0.2,
)


# Anhysteretic model parameters — shared across all plots in this notebook
props_anhysteretic = JAProps(
    ms=1.6e6,
    a=1100,
    alpha=0.0016,
    k=1.0,
    c=0.2,
)

m = 0.0  # initial (reference) magnetization for anhysteretic sweep

# Sweep the internal magnetic field to trace the anhysteretic curve
h_values = np.linspace(-70000, 70000, 801)
man_values = anhysteretic_magnetization(
    h=h_values,
    m=m,
    props=props,
)

# Validation point from the unit test
h_test = 300.0
man_expected = 144738.3548215022
man_test = anhysteretic_magnetization(
    h=h_test,
    m=m,
    props=props,
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(h_values, man_values, color='#2C5A9A',linewidth=2.5, label='Anhysteretic curve')
ax.scatter(
    [h_test],
    [man_test],
    color='crimson',
    s=80,
    zorder=3,
    label='Validation point at H = 300 A/m, M=0',
)
ax.axvline(h_test, color='crimson', linestyle='--', linewidth=1.2, alpha=0.7)
ax.axhline(man_expected, color='gray', linestyle=':', linewidth=1.2, alpha=0.8)

param_text = '\n'.join([
    f'Ms = {props.ms:.2e} A/m',
    f'a = {props.a:.0f} A/m',
    f'alpha = {props.alpha:.4f}',
])
ax.text(
    0.02,
    0.98,
    param_text,
    transform=ax.transAxes,
    fontsize=10,
    va='top',
    bbox=dict(boxstyle='round,pad=0.35', facecolor='white', alpha=0.85, edgecolor='0.5'),
)

ax.set_title('Anhysteretic Magnetization Curve')
ax.set_xlabel('Internal magnetic field H [A/m]')
ax.set_ylabel('Anhysteretic magnetization M_an [A/m]')
ax.grid(True, alpha=0.3)
ax.legend()

print(f'Computed M_an at H = {h_test:.0f} A/m: {man_test:.6f} A/m')
print(f'Expected M_an from test:           {man_expected:.6f} A/m')
print(f'Absolute error:                    {abs(man_test - man_expected):.6e} A/m')

plt.tight_layout()
plt.show()


Computed M_an at H = 300 A/m: 144738.354822 A/m
Expected M_an from test:           144738.354822 A/m
Absolute error:                    0.000000e+00 A/m


C:\Users\Rainy\AppData\Local\Temp\ipykernel_39328\4202069645.py:78: UserWarning: FigureCanvasCairo is non-interactive, and thus cannot be shown
  plt.show()


## 2. Plot the anhysteretic derivative

In [29]:
# Reuse the same sweep and props from the previous cell
h_values = np.linspace(-70000, 70000, 801)
dman_dh_values = anhysteretic_magnetization_differential(
    h=h_values,
    m=m,
    props=props,
)

# Validation point from dMan/dH limit and closed-form checks
h_test = 300.0
dman_dh_test = anhysteretic_magnetization_differential(
    h=h_test,
    m=m,
    props=props,
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(h_values, dman_dh_values, linewidth=2.5, color='#2C5A9A', label='dM_an/dH')
ax.scatter(
    [h_test],
    [dman_dh_test],
    color='darkorange',
    s=80,
    zorder=3,
    label='Derivative at H = 300 A/m',
)
ax.axvline(h_test, color='darkorange', linestyle='--', linewidth=1.2, alpha=0.7)

param_text = '\n'.join([
    f'Ms = {props.ms:.2e} A/m',
    f'a = {props.a:.0f} A/m',
    f'alpha = {props.alpha:.4f}',
])
ax.text(
    0.02,
    0.98,
    param_text,
    transform=ax.transAxes,
    fontsize=10,
    va='top',
    bbox=dict(boxstyle='round,pad=0.35', facecolor='white', alpha=0.85, edgecolor='0.5'),
)

ax.set_title('Anhysteretic Differential Curve')
ax.set_xlabel('Internal magnetic field H [A/m]')
ax.set_ylabel('dM_an/dH [-]')
ax.grid(True, alpha=0.3)
ax.legend()

print(f'Computed dM_an/dH at H = {h_test:.0f} A/m: {dman_dh_test:.6f}')
print(f'Origin limit Ms/(3a):                     {props.ms / (3.0 * props.a):.6f}')

plt.tight_layout()
plt.show()


Computed dM_an/dH at H = 300 A/m: 477.720139
Origin limit Ms/(3a):                     484.848485


C:\Users\Rainy\AppData\Local\Temp\ipykernel_39328\1270306141.py:54: UserWarning: FigureCanvasCairo is non-interactive, and thus cannot be shown
  plt.show()


## 3. Plot full hysteresis loop with anhysteretic overlay

In [30]:
# Full-loop Jiles-Atherton hysteresis trajectory using ja_solver
# Plot the initial magnetization curves to match Fig. (9) in [2]
h_max = 5000.0
solver_type = 2  # RK45

# Segment 1: 0 -> +Hmax
h1, m1 = ja_solver(
    props=props,
    h_start=0.0,
    h_end=h_max,
    m0=0.0,
    solver_type=solver_type,
)

# Segment 2: +Hmax -> -Hmax
h2, m2 = ja_solver(
    props=props,
    h_start=h_max,
    h_end=-h_max,
    m0=float(m1[-1]),
    solver_type=solver_type,
)

# Segment 3: -Hmax -> +Hmax (close the major loop)
h3, m3 = ja_solver(
    props=props,
    h_start=-h_max,
    h_end=h_max,
    m0=float(m2[-1]),
    solver_type=solver_type,
)

# Concatenate, removing duplicated segment boundary points
h_loop = np.concatenate([h1, h2[1:], h3[1:]])
m_loop = np.concatenate([m1, m2[1:], m3[1:]])

# Anhysteretic reference curve overlay (same H-span)
h_an, m_an = ja_solver(
    props=props_anhysteretic,
    h_start=-h_max,
    h_end=h_max,
    m0=float(m2[-1]),
    solver_type=solver_type,
)

# Scale magnetization to kA/m for clearer y-axis values
m_loop_ka = m_loop / 1000.0
m_an_ka = m_an / 1000.0

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(h_loop, m_loop_ka, color='#2C5A9A', linewidth=2.0, label='J-A hysteresis loop')
ax.plot(
    h_an,
    m_an_ka,
    color='black',
    linestyle='--',
    linewidth=2.0,
    alpha=0.9,
    label='Anhysteretic curve',
)

param_text = '\n'.join([
    f'Ms = {props.ms:.2e} A/m',
    f'a = {props.a:.0f} A/m',
    f'alpha = {props.alpha:.4f}',
    f'k = {props.k:.1f} A/m',
    f'c = {props.c:.3f}',
])
ax.text(
    0.02,
    0.98,
    param_text,
    transform=ax.transAxes,
    fontsize=10,
    va='top',
    bbox=dict(boxstyle='round,pad=0.35', facecolor='white', alpha=0.85, edgecolor='0.5'),
)

ax.set_title('Jiles-Atherton Hysteresis with Anhysteretic Overlay')
ax.set_xlabel('Internal magnetic field H [A/m]')
ax.set_ylabel('Magnetization M [kA/m]')
ax.grid(True, alpha=0.3)
ax.legend()

print(f'Loop points: {h_loop.size}')
print(f'M range: [{np.min(m_loop_ka):.2f}, {np.max(m_loop_ka):.2f}] kA/m')

plt.tight_layout()
plt.show()


Loop points: 311
M range: [-1337.13, 1337.13] kA/m


C:\Users\Rainy\AppData\Local\Temp\ipykernel_39328\1300957888.py:89: UserWarning: FigureCanvasCairo is non-interactive, and thus cannot be shown
  plt.show()


## 4. Plot full hysteresis loop as magnetic flux density B

In [31]:
# Convert the full hysteresis loop from magnetization to flux density
b_loop = MU0 * (h_loop + m_loop)
b_ref = MU0 * (h_an + m_an)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(h_loop, b_loop, color='#2C5A9A', linewidth=2.0, label='J-A hysteresis loop (B)')
ax.plot(
    h_an,
    b_ref,
    color='black',
    linestyle='--',
    linewidth=1.8,
    alpha=0.85,
    label='Anhysteretic reference (B)',
)

param_text = '\n'.join([
    f'Ms = {props.ms:.2e} A/m',
    f'a = {props.a:.0f} A/m',
    f'alpha = {props.alpha:.4f}',
    f'k = {props.k:.1f} A/m',
    f'c = {props.c:.3f}',
])
ax.text(
    0.02,
    0.98,
    param_text,
    transform=ax.transAxes,
    fontsize=10,
    va='top',
    bbox=dict(boxstyle='round,pad=0.35', facecolor='white', alpha=0.85, edgecolor='0.5'),
)

ax.set_title('Jiles-Atherton Hysteresis as Flux Density')
ax.set_xlabel('Internal magnetic field H [A/m]')
ax.set_ylabel('Magnetic flux density B [T]')
ax.grid(True, alpha=0.3)
ax.legend()

print(f'B range: [{np.min(b_loop):.4f}, {np.max(b_loop):.4f}] T')

plt.tight_layout()
plt.show()


B range: [-1.6866, 1.6866] T


C:\Users\Rainy\AppData\Local\Temp\ipykernel_39328\986091933.py:43: UserWarning: FigureCanvasCairo is non-interactive, and thus cannot be shown
  plt.show()


## 5. Stress sweep: B-H hysteresis shift vs stress

This section sweeps stress \(\sigma_0\) and plots the full hysteresis loop as magnetic flux density:

\[
B = \mu_0(H + M)
\]

Stress enters through the effective field term used by the model.

In [32]:
# Function to sweep stress and plot B-H hysteresis loops for a given JAProps class

def plot_stress_sweep_bh(JAPropsClass,
                        gamma1_intercept_ref=4.0e-18,
                        gamma2_intercept_ref=-2.0e-30,
                        gamma1_sigma_slope=-2.0e-26,
                        gamma2_sigma_slope=+5.0e-39,
                        stress_values_mpa=[-100.0, 0.0, 100.0],
                        theta_deg=0.0,
                        nu=0.30,
                        h_max_local=15000.0,
                        solver_type_local=3):
    """
    Sweep stress and plot B-H hysteresis loops using the provided JAProps class.
    Args:
        JAPropsClass: The JAProps class to instantiate property objects.
        gamma1_intercept_ref: float, gamma1 intercept at sigma0=0
        gamma2_intercept_ref: float, gamma2 intercept at sigma0=0
        gamma1_sigma_slope: float, slope of gamma1 vs stress
        gamma2_sigma_slope: float, slope of gamma2 vs stress
        stress_values_mpa: list of stress values in MPa
        theta_deg: float, angle in degrees
        nu: float, Poisson's ratio
        h_max_local: float, max H field for sweep
        solver_type_local: int, solver type for ja_solver
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    custom_colors = ['#2C5A9A', '#000000', '#E13C30']
    for color, sigma_mpa in zip(custom_colors, stress_values_mpa):
        sigma_pa = sigma_mpa * 1.0e6
        props_sigma = JAPropsClass(
            ms=1.7e6,
            a=1000,
            alpha=0.001,
            k=10,
            c=0.1,
            theta=np.deg2rad(theta_deg),
            nu=nu,
            sigma0=0.0,
            gamma1_intercept=gamma1_intercept_ref,
            gamma2_intercept=gamma2_intercept_ref,
            gamma1_sigma_slope=gamma1_sigma_slope,
            gamma2_sigma_slope=gamma2_sigma_slope,
        )
        props_sigma.sigma0 = sigma_pa

        # Solve one full loop in three monotonic segments
        h1_s, m1_s = ja_solver(
            props=props_sigma,
            h_start=0.0,
            h_end=h_max_local,
            m0=0.0,
            solver_type=solver_type_local,
        )
        h2_s, m2_s = ja_solver(
            props=props_sigma,
            h_start=h_max_local,
            h_end=-h_max_local,
            m0=float(m1_s[-1]),
            solver_type=solver_type_local,
        )
        h3_s, m3_s = ja_solver(
            props=props_sigma,
            h_start=-h_max_local,
            h_end=h_max_local,
            m0=float(m2_s[-1]),
            solver_type=solver_type_local,
        )

        h_loop_s = np.concatenate([h1_s, h2_s[1:], h3_s[1:]])
        m_loop_s = np.concatenate([m1_s, m2_s[1:], m3_s[1:]])
        b_loop_s = MU0 * (h_loop_s + m_loop_s)

        ax.plot(
            h_loop_s,
            b_loop_s,
            color=color,
            linewidth=2.0,
            label=f"sigma0 = {sigma_mpa:.0f} MPa",
        )

    # Fix: Remove math mode from param_text and ensure no $ in any string
    param_text = '\n'.join([
        f'Ms = {props_sigma.ms:.2e} A/m',
        f'a = {props_sigma.a:.0f} A/m',
        f'alpha = {props_sigma.alpha:.4f}',
        f'k = {props_sigma.k:.1f} A/m',
        f'c = {props_sigma.c:.3f}',
        f'theta = {theta_deg:.1f} deg',
        f'nu = {nu:.2f}',
        f'gamma1(0) = {props_sigma.gamma1:.2e}',
        f'gamma2(0) = {props_sigma.gamma2:.2e}',
    ])
    ax.text(
        0.02,
        0.98,
        param_text,
        transform=ax.transAxes,
        fontsize=9,
        va='top',
        bbox=dict(boxstyle='round,pad=0.35', facecolor='white', alpha=0.85, edgecolor='0.5'),
    )

    ax.set_title('Stress Sweep: B-H Hysteresis Shift')
    ax.set_xlabel('Internal magnetic field H [A/m]')
    ax.set_ylabel('Magnetic flux density B [T]')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right')

    print('Stress sweep complete for sigma0 values [MPa]:', stress_values_mpa)
    # Fix: Remove plt.tight_layout() to avoid ValueError with Cairo backend
    # plt.tight_layout()

    # Save as PDF with smart filename
    save_plot_pdf(fig, '5.0', 'Anhysteretic_Stress_Sweep')

    plt.show()


In [33]:
# Example usage:
matplotlib.use('cairo')

plot_stress_sweep_bh(JAProps)

Stress sweep complete for sigma0 values [MPa]: [-100.0, 0.0, 100.0]
✓ PDF saved: plots_pdf\section_5_0_anhysteretic_stress_sweep.pdf


C:\Users\Rainy\AppData\Local\Temp\ipykernel_39328\3510404024.py:117: UserWarning: FigureCanvasCairo is non-interactive, and thus cannot be shown
  plt.show()
